# AI-Driven Threat Detection & Anomaly Classification
## Proyek Web E-Commerce Keripik Kentang (Kentang Emas)
**Mata Kuliah:** Perancangan Keamanan Sistem dan Jaringan (INF11022) - Fase 4  
**Algoritma:** Isolation Forest (Unsupervised) & Random Forest (Supervised)  
**Tujuan:** Mendeteksi anomali autentikasi (Brute Force, Credential Stuffing, Botnet Scan) dari log keamanan.

In [1]:
import numpy as np
import pandas as pd
import matplotlib.pyplot as plt

# 1. Load Security Log Dataset
df = pd.read_csv('auth_security_logs.csv')
print(f"Total data log: {len(df)} baris")
df.head()

### 2. Feature Engineering & Train-Test Split
Fitur yang diekstraksi dari log autentikasi:
- `login_hour`: Jam aktivitas login (0 - 23)
- `failed_attempts_5m`: Frekuensi login gagal dalam jendela waktu 5 menit
- `request_rate_per_min`: Volume request HTTP per menit
- `ip_entropy`: Entropi keragaman alamat IP
- `payload_risk_score`: Deteksi karakter berbahaya (SQLi / XSS)

In [2]:
features = ['login_hour', 'failed_attempts_5m', 'request_rate_per_min', 'ip_entropy', 'payload_risk_score']
X = df[features].values
y = df['is_threat'].values

# Split 75% Training, 25% Testing
np.random.seed(42)
indices = np.random.permutation(len(X))
split_idx = int(len(X) * 0.75)
X_train, X_test = X[indices[:split_idx]], X[indices[split_idx:]]
y_train, y_test = y[indices[:split_idx]], y[indices[split_idx:]]
print(f"Data Train: {len(X_train)}, Data Test: {len(X_test)}")

### 3. Training Model Isolation Forest & Random Forest

In [3]:
# Eksekusi training script pendeteksi anomali
from train_ai_detector import CustomIsolationForest, SimpleDecisionTree, calculate_metrics

iso_forest = CustomIsolationForest(n_trees=100, max_samples=256)
iso_forest.fit(X_train)

scores_test = iso_forest.predict_score(X_test)
y_pred_iso = (scores_test >= np.percentile(scores_test, 88)).astype(int)

rf_model = SimpleDecisionTree().fit(X_train, y_train)
y_pred_rf = rf_model.predict(X_test).astype(int)

print("Training selesai!")

### 4. Evaluasi Kinerja Model (Precision, Recall, F1-Score, False Positives)

In [4]:
m_iso = calculate_metrics(y_test, y_pred_iso)
m_rf = calculate_metrics(y_test, y_pred_rf)

results = pd.DataFrame({
    'Isolation Forest': [m_iso['Accuracy'], m_iso['Precision'], m_iso['Recall'], m_iso['F1-Score'], m_iso['FPR']],
    'Random Forest': [m_rf['Accuracy'], m_rf['Precision'], m_rf['Recall'], m_rf['F1-Score'], m_rf['FPR']]
}, index=['Accuracy', 'Precision', 'Recall', 'F1-Score', 'False Positive Rate (FPR)'])

print(results * 100)

### 5. Visualisasi Hasil Deteksi Anomali

In [5]:
# Menampilkan grafik sebaran anomali
plt.figure(figsize=(10, 5))
plt.scatter(X_test[y_test == 0, 1], X_test[y_test == 0, 2], c='#2563EB', alpha=0.6, label='Trafik Normal')
plt.scatter(X_test[y_test == 1, 1], X_test[y_test == 1, 2], c='#DC2626', alpha=0.8, marker='^', label='Ancaman / Anomali')
plt.title('Deteksi Anomali AI: Gagal Login vs Request Rate')
plt.xlabel('Gagal Login (5 menit)')
plt.ylabel('Request Rate (req/min)')
plt.legend()
plt.grid(True, linestyle='--', alpha=0.5)
plt.show()